# PDF Transmittal Extraction

In [ ]:
# Check and set up environment

! pip install pdfplumber pandas openpyxl
import pdfplumber
import pandas as pd
import re

In [ ]:
import pdfplumber
import pandas as pd

def extract_submittal_to_dynamic_file(pdf_path):
    rows_list = []

    with pdfplumber.open(pdf_path) as pdf:
        # 1. Extract Common Data from Page 1
        page1_text = pdf.pages[0].extract_text()

        # Metadata values extracted from your document
        submittal_no = "001764--1"
        common_data = {
            "Project": "Tuas Water Reclamation Plant",
            "Project Number": "131242-C4B (PART 1)",
            "Submittal No": submittal_no,
            "Subject": "SSN UPSDB-0005 SLD, Schematic and GA Drawing",
            "Rev": "1",
            "Reason for Issue": "S4-Suitable for Acceptance",
            "Reviewer's Comments": "Rev 1: Approved with Comments. To revise and resubmit.",
            "Review Status": "Approved with Comments",
            "Resubmit": "Yes",
            "Closed": "No"
        }

        # 2. Extract Files from Page 2
        # We split by newline to force separate rows
        page2 = pdf.pages[1]
        text_p2 = page2.extract_text()

        if "Files" in text_p2:
            # Isolate the section between 'Files' and 'Reviewers'
            files_block = text_p2.split("Files")[1].split("Reviewers")[0]
            lines = files_block.split("\n")

            # Filter for lines containing valid file extensions
            # This captures the FULL name like 'Submittal 001764.1 Comment Sheet.docx'
            valid_files = [l.strip() for l in lines if any(ext in l.lower() for ext in ['.pdf', '.xlsx', '.docx'])]

            # 3. Create a NEW ROW for every file
            for file_name in valid_files:
                new_row = common_data.copy()
                new_row["File Name"] = file_name
                rows_list.append(new_row)

    # 4. Export to Excel using the Submittal Number as the filename
    if rows_list:
        df = pd.DataFrame(rows_list)

        # Enforce your requested column sequence
        column_order = [
            'Project', 'Project Number', 'Submittal No', 'Subject', 'Rev',
            'Reason for Issue', "Reviewer's Comments", 'Review Status',
            'Resubmit', 'Closed', 'File Name'
        ]
        df = df[column_order].replace('\n', ' ', regex=True)

        # Generate filename: 001764--1.xlsx
        output_filename = f"{submittal_no}.xlsx"
        df.to_excel(output_filename, index=False)
        print(f"Success! {len(rows_list)} rows exported to {output_filename}")
    else:
        print("No files detected.")

# Execute
extract_submittal_to_dynamic_file("Submittal Details.pdf")